# 08 — CP1 B3: timestamp conflict và ảnh hưởng tới stay

Notebook độc lập: mở từ đầu, không cần chạy lại NB06/NB07. B2 đã có corpus selection.
B3 so sáu policy cleaning trên **toàn bộ GeoLife thật**, giữ detector fixed-anchor và config baseline.
Xem protocol tại `reports/cp1/cleaning_protocol.md`. Đây là sensitivity/audit, không phải accuracy.

Cell thiết lập đã bật `RUN_B3=True` để bạn chủ động chạy lần đầu. Job đọc raw PLT đủ mọi user,
đối chiếu baseline và lưu output riêng; có thư mục kết quả sẽ từ chối ghi đè. Chưa có số đo runtime B3.


In [ ]:
from pathlib import Path
import sys, json, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd().resolve()
if not (ROOT/'pyproject.toml').exists(): ROOT = ROOT.parent
if str(ROOT/'src') not in sys.path: sys.path.insert(0,str(ROOT/'src'))
import geolife_inference.benchmark_cleaning as cleaning
cleaning = importlib.reload(cleaning)
OUT = ROOT/'artifacts/benchmark/b3/cleaning_v1'
RUN_B3 = True
ADDITIONAL_COMPUTE_SECONDS = 0.0  # Cộng job mới/failed nếu có sau B2; giữ 0 nếu chưa có.
b2_audit = json.loads((ROOT/'artifacts/benchmark/b2/selection_v1/generation_audit.json').read_text(encoding='utf-8'))
if b2_audit['status'] != 'completed': raise ValueError('B2 selection chưa completed')
USED_SECONDS = b2_audit['compute_used_after_seconds'] + ADDITIONAL_COMPUTE_SECONDS
print('Output:',OUT)
print('Compute ledger trước B3:',USED_SECONDS,'giây, gồm reserve khởi đầu đã công bố')
print('Policies:',cleaning.POLICIES)


## 1. Quy tắc so sánh

- vote10/20/50 khác ngưỡng không gian cho cùng timestamp voting/context policy.
- mean20 chỉ gộp nếu mọi cặp ≤20 m; no_merge quarantine mọi nhóm multi-position và ngắt segment.
- duplicate_altitude đổi duy nhất duplicate key có thêm altitude parsed, giữ vote20.
- Detector luôn 100 m/10 phút, max gap 5 phút. Không chạy grouping hoặc Home/Office ở bước này.
- Trajectory không chịu tác động được tái dùng vote20, có manifest rõ; mọi user đều được kiểm kê.

Mỗi user phải khớp vote20 với preprocessing counters và stays saved trước khi được ghi completed.
Raw row ID/file hash dùng cho provenance; local segment/cluster IDs không dùng để ghép event.


In [ ]:
if RUN_B3:
    result = cleaning.run(ROOT, OUT, USED_SECONDS)
    print('B3:',result['status'])
else:
    print('B3 tắt; đọc kết quả hiện có bên dưới.')


## 2. Audit và counts toàn dataset

Chỉ đọc như kết quả đầy đủ khi audit completed và đủ 182 users, 18.670 trajectories,
24.876.978 raw points. Nếu failed, đọc error, không chạy lại vào cùng namespace.


In [ ]:
audit_path = OUT/'audit.json'
if not audit_path.exists(): raise FileNotFoundError('Chưa có B3 audit')
audit = json.loads(audit_path.read_text(encoding='utf-8'))
print('Status:',audit['status'])
if audit['status'] != 'completed': raise RuntimeError(audit.get('error','Job chưa completed'))
assert len(audit['completed_users']) == 182
assert len(audit['baseline_parity_users']) == 182
assert audit['processed_trajectories'] == 18_670
assert audit['raw_points'] == 24_876_978
print('Runtime:',audit['elapsed_seconds'],'giây; sampled peak WS:',audit['sampled_peak_working_set_bytes']/1024**2,'MiB')
totals = pd.read_csv(OUT/'counts_totals.csv').set_index('policy').reindex(cleaning.POLICIES)
display(totals[['input_points','output_points','invalid_rows_removed','duplicate_records_removed',
    'conflict_groups','merged_extra_points','unresolved_conflict_rows','context_resolved_groups','breaks_due_to_conflict']])


## 3. Stay-diff và dwell

Graph ghép theo cùng user/trajectory, overlap thời gian dương; chia connected components để thấy
split/merge. Membership hash phản ánh raw contributor IDs. `unchanged` cần khớp cả membership,
point_count, thời gian và tọa độ trong dung sai đã khóa. Counts trên biểu đồ là số components;
không diễn giải số stay nhiều hơn là chất lượng tốt hơn.


In [ ]:
summary = pd.read_csv(OUT/'stay_diff_per_user.csv',dtype={'user_id':str})
kind_cols = ['unchanged','changed','lost','new','split','merge','complex']
by_policy = summary.groupby('policy')[kind_cols+['baseline_stays','variant_stays','baseline_dwell_minutes','variant_dwell_minutes']].sum().reindex(cleaning.POLICIES)
by_policy['dwell_delta_hours'] = (by_policy.variant_dwell_minutes-by_policy.baseline_dwell_minutes)/60
affected = summary.assign(affected=summary[kind_cols[1:]].sum(axis=1)>0).groupby('policy').affected.sum()
by_policy['affected_users'] = affected
display(by_policy)
fig, axes = plt.subplots(1,2,figsize=(14,5))
by_policy[kind_cols[1:]].plot.bar(stacked=True,ax=axes[0])
axes[0].set(title='Components khác vote20',xlabel='Cleaning policy',ylabel='Số components')
by_policy.dwell_delta_hours.plot.bar(ax=axes[1],color='steelblue')
axes[1].axhline(0,color='black',linewidth=.8)
axes[1].set(title='Thay đổi tổng dwell so với vote20',xlabel='Cleaning policy',ylabel='Giờ dwell chênh lệch')
plt.tight_layout();plt.show()
print('Quyết định:',audit['decision'])
print('Counts/stay-diff không chứng minh policy nào có tọa độ đúng hơn; giữ vote20 trừ lỗi audit cụ thể.')


## 4. Đọc kết quả trước B4

Gửi audit và bảng tổng hợp để cùng xem policy nào làm thay đổi stays và vì sao. Có thể truy vết
`stay_diff.csv`, CSV stays từng user/policy và conflict provenance; các nhánh reused_vote20 tham chiếu
provenance vote20. Không tick B3 chỉ vì code đã có; cần audit/parity full-data hoàn tất.

B4 mới chạy năm detector trên bộ tổng hợp B2 cùng grid đã chốt. B3 giữ policy baseline nếu chưa phát hiện lỗi.
